# Aula 14A · Automação de equipamentos

Esta semana apresenta o [capítulo 14 do site](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/). A ideia central: **o mesmo comando em N
equipamentos, sem esquecer nenhum e sem parar no primeiro que falha**. Conectar, mandar
o comando, transformar a saída (texto) em dado, registrar quem não respondeu — e montar
o relatório.

**Ao fim das duas noites você consegue:**

1. conectar num equipamento, mandar um comando e ler a saída;
2. transformar a saída de um comando em dado com as ferramentas da Aula 09;
3. escrever o coletor que percorre o inventário e registra as falhas sem parar.

**Noite A — a ideia** (1h40): 📝 mini-teste · 📟 chamado · 1. conectar e mandar um
comando · 2. `with` · 3. da saída ao dado · 4. o coletor completo · 5. do resultado
ao relatório · 6. com equipamentos de verdade · 🚪 antes de sair

O chamado se resolve na [noite
B](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/aula14b-automacao.ipynb),
no encontro seguinte, que é laboratório: nenhum assunto novo, os 🎯 que ficaram e a
lista começada em sala.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

A célula ⚙️ desta aula também **põe no ar os switches simulados**, com a mesma
interface da biblioteca Netmiko. Usuário `noc`, senha `marenet`. Os switches
`10.0.1.20`, `10.0.2.20` e `10.0.5.20` respondem; o `10.0.3.20` não responde e o
`10.0.4.20` recusa a senha — de propósito.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
# simulador: equipamentos SSH da Maré Net — não precisa ler (faz o papel dos switches).
# Imita a interface da biblioteca Netmiko: ConnectHandler(...), send_command(...),
# disconnect() e as mesmas exceções. Com equipamentos de verdade, a única linha que
# muda é o import: from netmiko import ConnectHandler, ...
import time


class NetmikoTimeoutException(Exception):
    """O equipamento não respondeu a tempo."""


class NetmikoAuthenticationException(Exception):
    """Usuário ou senha recusados."""


_VERSAO = """Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version {versao}, RELEASE SOFTWARE (fc3)
{nome} uptime is {uptime}
System image file is "flash:c2960x-universalk9-mz.{versao}.bin"
cisco WS-C2960X-48FPD-L (APM86XXX) processor with 524288K bytes of memory."""

_CABECALHO = "Interface              IP-Address      OK? Method Status                Protocol"

_EQUIPAMENTOS = {
    "10.0.1.20": {"nome": "SWITCH-CENTRO-01", "versao": "15.2(7)E7", "uptime": "3 weeks, 2 days, 4 hours",
                  "interfaces": [("Vlan10", "10.0.1.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down")]},
    "10.0.2.20": {"nome": "SWITCH-NORTE-02", "versao": "15.2(7)E7", "uptime": "12 days, 7 hours",
                  "interfaces": [("Vlan20", "10.0.2.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "administratively down", "down"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down"),
                                 ("GigabitEthernet1/0/4", "unassigned", "down", "down")]},
    "10.0.3.20": {"nome": "SWITCH-SUL-03", "falha": "tempo"},
    "10.0.4.20": {"nome": "SWITCH-LESTE-04", "falha": "senha"},
    "10.0.5.20": {"nome": "SWITCH-OESTE-05", "versao": "15.2(4)E10", "uptime": "1 year, 5 weeks",
                  "interfaces": [("Vlan50", "10.0.5.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up")]},
}
_SENHA = "marenet"


class _Conexao:
    def __init__(self, dados):
        self._dados = dados

    def send_command(self, comando):
        comando = " ".join(comando.split())
        if comando == "show version":
            return _VERSAO.format(**self._dados)
        if comando == "show ip interface brief":
            linhas = [_CABECALHO]
            for nome, ip, estado, protocolo in self._dados["interfaces"]:
                linhas.append(f"{nome:<23}{ip:<16}YES manual {estado:<22}{protocolo}")
            return "\n".join(linhas)
        if comando == "show clock":
            return "*14:03:17.123 BRT Mon Mar 2 2026"
        return "% Invalid input detected at '^' marker."

    def disconnect(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.disconnect()


def ConnectHandler(device_type, host, username, password, timeout=5, **extras):
    """Abre uma "sessão SSH" com o equipamento simulado."""
    dados = _EQUIPAMENTOS.get(host)
    if dados is None or dados.get("falha") == "tempo":
        time.sleep(0.2)
        raise NetmikoTimeoutException(f"TCP connection to device failed: {host}")
    if dados.get("falha") == "senha" or password != _SENHA:
        raise NetmikoAuthenticationException(f"Authentication to device failed: {host}")
    return _Conexao(dados)


print("equipamentos simulados prontos (senha do usuário noc: marenet)")

## 📝 Mini-teste — dez minutos

Caderno fechado. O **Mini-teste 14**, no papel, cobre as duas noites da Aula 13:
rastrear um trecho curto e escrever uma função pequena. Depois dele, o chamado de
hoje.

> 📡 **Na rede: switch, SSH e terminal.**
>
> Um **switch** é o equipamento que liga vários outros num mesmo local, com uma
> **interface** (porta) para cada cabo. Para configurar ou consultar um switch, o
> técnico abre o **terminal** dele — uma tela de texto onde se digitam comandos —
> pela rede, com o **SSH**, que é um acesso remoto seguro (a conversa vai
> criptografada, como uma ligação que ninguém consegue grampear). Os comandos que
> começam com `show` ("mostre") só **leem** informações. Veja
> [Conversar com os equipamentos](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/#conversar-com-os-equipamentos-servidor-api-e-ssh).

## 📟 O chamado de hoje

> **Chamado #1609 — NOC Maré Net**
>
> *"Estagiário, vamos atualizar o firmware dos switches para a versão **15.2(7)E7**.
> Preciso saber **quais estão em outra versão** e **quais nem responderam** — esses
> vão para a equipe de campo. Hoje alguém entra em um por um pelo terminal e anota
> numa planilha."*

Na noite B o script faz a volta nos switches sozinho.

## 1. Conectar e mandar um comando

`ConnectHandler(...)` abre a sessão SSH; `send_command("...")` manda o comando e
devolve **a saída inteira, como texto**; `disconnect()` fecha. É a interface da
Netmiko, a biblioteca mais usada para isso.

📖 [capítulo 14 · Conectar e mandar um comando](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#conectar-e-mandar-um-comando)

> 💡 **Pense assim: o telefonema.**
>
> Conectar é **discar** para o equipamento e ele atender (o `ConnectHandler`, com o
> endereço e a senha); `send_command` é **falar** uma instrução e ouvir a resposta
> inteira; `disconnect` é **desligar**. Enquanto a ligação está aberta, ela ocupa
> uma das poucas linhas do equipamento.

**✍️ Passo 1.** Faça `conexao = ConnectHandler(device_type="cisco_ios", host="10.0.1.20",
username="noc", password="marenet")`. Depois `saida = conexao.send_command("show version")`,
imprima `type(saida)` e `saida`, e feche com `conexao.disconnect()`.

In [ ]:
# ✍️ passo 1

**Preveja:** de que tipo é o que o `send_command` devolve?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`<class 'str'>` e quatro linhas de texto — exatamente o que apareceria no terminal. O
equipamento responde em **texto**; transformar texto em dado é trabalho do script, com
as ferramentas da Aula 09.

</details>

## 2. `with`: a conexão que fecha sozinha

O `with` é o mesmo do `open()`: ao sair do bloco — inclusive por causa de um erro —, a
conexão fecha. E `**dispositivo` desempacota um dicionário em argumentos com nome.

📖 [capítulo 14 · `with`: a conexão que fecha sozinha](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#with-a-conexao-que-fecha-sozinha)

> 💡 **Pense assim: a porta com mola.**
>
> O `with` é a porta com mola da loja: você entra, faz o que tem de fazer, e ao sair
> ela fecha sozinha — mesmo que você saia correndo. Sem ela, alguém sempre esquece a
> porta aberta.

> 💡 **Pense assim: o formulário preenchido com a ficha.**
>
> O `**` pega a ficha (o dicionário) e preenche o formulário (a função) campo a campo:
> o que está em `"host"` na ficha vai para o campo `host` do formulário, e assim por
> diante. Os nomes das chaves precisam ser exatamente os nomes que a função espera.

In [ ]:
# 📦 dados prontos — só rode esta célula
dispositivo = {"device_type": "cisco_ios", "host": "10.0.2.20",
               "username": "noc", "password": "marenet"}

**✍️ Passo 2.** Escreva `with ConnectHandler(**dispositivo) as conexao:` e, dentro, imprima
`conexao.send_command("show clock")` e `conexao.send_command("show ip interface brief")`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que o `**dispositivo` faz com o dicionário?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Passa cada chave como um argumento com nome — é o mesmo que escrever
`device_type="cisco_ios", host="10.0.2.20", ...`. Sai a hora do equipamento e a tabela
das interfaces. Guardar os dados de acesso num dicionário (vindo do inventário) e passar
com `**` é a forma que se vê em todo código de automação.

</details>

> ⚠️ **Armadilha.** Abrir conexão e esquecer de fechar. Equipamento de rede aceita poucas sessões ao mesmo
> tempo; um script que deixa sessões abertas acaba bloqueando o acesso da própria equipe.
> Com `with`, fechar não depende de lembrar.

## 3. Da saída ao dado

A saída do `show ip interface brief` é uma tabela em texto, e o estado pode ter espaço
dentro (`administratively down`). `split(maxsplit=4)` separa as quatro primeiras colunas
e deixa o resto inteiro; `rpartition(" ")` — o `partition` que corta na **última**
ocorrência — separa o protocolo do estado.

📖 [capítulo 14 · Da saída ao dado](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#da-saida-ao-dado)

> 📡 **Na rede: interface, estado e protocolo.**
>
> Cada linha do `show ip interface brief` é uma **interface** (porta) do switch:
> `GigabitEthernet1/0/1` é a porta 1 do módulo 0 da unidade 1, com velocidade de 1
> Gbit/s; `Vlan20` é uma interface virtual. O **estado** diz se a porta está ligada
> na parte física (`up`, `down`) ou se alguém a desligou de propósito
> (`administratively down`); o **protocolo** diz se ela está conversando com o
> equipamento do outro lado.

In [ ]:
# 📦 dados prontos — só rode esta célula
saida = """Interface              IP-Address      OK? Method Status                Protocol
Vlan20                 10.0.2.20       YES manual up                    up
GigabitEthernet1/0/1   unassigned      YES manual up                    up
GigabitEthernet1/0/2   unassigned      YES manual administratively down down
GigabitEthernet1/0/3   unassigned      YES manual down                  down"""

**✍️ Passo 3.** Crie `estados = {}` e percorra `saida.splitlines()[1:]`. Em cada linha, faça
`interface, ip, ok, metodo, resto = linha.split(maxsplit=4)` e
`estado, _, protocolo = resto.rpartition(" ")`; guarde
`estados[interface] = estado.strip()`. Imprima `estados`.

In [ ]:
# ✍️ passo 3

**Preveja:** por que o `[1:]`? E o que sai para a `GigabitEthernet1/0/2`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

O `[1:]` pula o cabeçalho. A `GigabitEthernet1/0/2` sai `'administratively down'`,
inteiro — com um `split()` sem limite, o estado viraria só `administratively`, e o
protocolo pularia para o lugar errado.

</details>

### 🎯 Sua vez — Os IPs das interfaces

Escreva `ips_configurados(saida)`, que devolve um dicionário **interface → IP** só das
interfaces que têm IP (as outras aparecem como `unassigned`).

In [ ]:
def ips_configurados(saida):
    # sua solução aqui
    pass

In [ ]:
confere(ips_configurados, [
    ((saida,), {"Vlan20": "10.0.2.20"}),
    (("Interface IP-Address OK? Method Status Protocol\nVlan1 10.0.9.1 YES manual up up\nVlan2 10.0.9.2 YES manual up up",),
     {"Vlan1": "10.0.9.1", "Vlan2": "10.0.9.2"}),
])

<details>
<summary><b>💡 Dica</b></summary>

O mesmo laço do passo, com `linha.split(maxsplit=4)`; guarde só **se** o `ip` for
diferente de `"unassigned"`.

</details>

## 4. O coletor completo

As peças juntas: um laço sobre o inventário e, para cada equipamento, uma função que
conecta, lê o que interessa e **nunca levanta erro** — cada falha vira um resultado
com o motivo. O resultado de cada um vai numa dataclass (Aula 10).

📖 [capítulo 14 · O coletor completo](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#o-coletor-completo)

> 💡 **Pense assim: o carteiro.**
>
> O carteiro não volta para a agência porque ninguém atendeu numa casa: deixa um
> aviso ("tentativa de entrega: ninguém em casa") e segue para a próxima. No fim do
> dia, a lista dele diz o que foi entregue e o que não foi, e por quê. O `Resultado`
> com `ok=False` e o motivo é esse aviso.

**✍️ Passo 4.** Escreva `versao_de(saida)`: percorra as linhas da saída; na que tem `"Version "`, faça
`_, _, depois = linha.partition("Version ")` e `versao, _, _ = depois.partition(",")`,
e devolva `versao`. Teste com a saída do `show version` do `10.0.1.20`.

In [ ]:
# ✍️ passo 4

**Preveja:** que versão sai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`15.2(7)E7`. São dois `partition` da Aula 09: um corta antes da versão, o outro depois
da vírgula.

</details>

**✍️ Passo 5.** Tente conectar no `10.0.3.20` — **sem** `try`.

In [ ]:
# ✍️ passo 5

**Preveja:** o que acontece com um equipamento que não responde?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`NetmikoTimeoutException: TCP connection to device failed: 10.0.3.20`. Num laço de
cinquenta switches, esse erro no terceiro encerraria a coleta — e os outros 47 ficariam
sem resposta.

</details>

**✍️ Passo 6.** Escreva `coleta(host)`: dentro de um `try:`, o `with ConnectHandler(...)` que lê a
versão com `versao_de(conexao.send_command("show version"))` e `return versao`;
`except NetmikoTimeoutException: return "tempo esgotado"`;
`except NetmikoAuthenticationException: return "senha recusada"`. Chame para os cinco
hosts, de `10.0.1.20` a `10.0.5.20`, imprimindo host e resultado.

In [ ]:
# ✍️ passo 6

**Preveja:** o laço chega ao `10.0.5.20`, mesmo com as falhas no meio?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Chega: três versões e duas falhas com o motivo — `tempo esgotado` para o `10.0.3.20` e
`senha recusada` para o `10.0.4.20`. Todo o trabalho com o equipamento está **dentro**
do `try`, e cada falha tem o seu `except`: o laço **nunca vê uma exceção**.

</details>

## 5. Do resultado ao relatório

Com os resultados numa lista de dicionários, o pandas da Aula 11 assume:
`value_counts` para contar versões, o filtro para listar as falhas, `to_csv` para
gravar.

📖 [capítulo 14 · Do resultado ao relatório](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#do-resultado-ao-relatorio)

In [ ]:
# 📦 dados prontos — só rode esta célula
linhas = [
    {"nome": "SWITCH-CENTRO-01", "ok": True, "versao": "15.2(7)E7"},
    {"nome": "SWITCH-NORTE-02", "ok": True, "versao": "15.2(7)E7"},
    {"nome": "SWITCH-SUL-03", "ok": False, "versao": ""},
    {"nome": "SWITCH-OESTE-05", "ok": True, "versao": "15.2(4)E10"},
]

**✍️ Passo 7.** Escreva `import pandas as pd`, faça `tabela = pd.DataFrame(linhas)` e imprima
`tabela[tabela["ok"]]["versao"].value_counts()`.

In [ ]:
# ✍️ passo 7

**Preveja:** o que o `tabela[tabela["ok"]]` faz antes do `value_counts`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Filtra só os que deram certo (a coluna `ok` já é a máscara) e conta as versões: dois na
`15.2(7)E7`, um na `15.2(4)E10`. É a primeira pergunta de quem planeja uma atualização
de firmware.

</details>

## 6. Com equipamentos de verdade

O simulador tem a mesma interface da Netmiko, inclusive os nomes das exceções. Com
equipamentos de verdade, o coletor muda **uma linha** — o `import`
(`from netmiko import ConnectHandler, ...`). E dois cuidados: a senha não vai escrita
no código, e a automação começa pelos comandos que **só leem**.

📖 [capítulo 14 · Com equipamentos de verdade](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#com-equipamentos-de-verdade)

> 💡 **Pense assim: a chave debaixo do tapete.**
>
> Senha escrita no código é a chave debaixo do tapete: qualquer um que olhar o
> arquivo — um colega, um repositório compartilhado, um *print* de tela — leva a
> chave de todos os equipamentos. A senha fica guardada à parte, e o script a pede na
> hora de rodar.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** O `send_command("show version")` devolve:
a) um dicionário com os campos  b) a saída do comando, como texto  c) `True` se deu
certo  d) uma lista de linhas

> 🌉 **Esta fica sem resposta aqui.** Pense nela até o encontro seguinte: é a
> primeira coisa da noite B.

**2.** Na coleta de 50 switches, o terceiro não responde. Com o `try` **dentro** do
laço, o que acontece?
a) a coleta para no terceiro  b) o terceiro é registrado como falha e os outros 47 são
coletados  c) o script tenta o terceiro para sempre  d) todos falham

<details>
<summary><b>Resposta da 2</b></summary>

**b**. Com o `try` em volta do laço inteiro, seria a **a**.

</details>

**3.** Para usar um equipamento de verdade em vez do simulador, o que muda no coletor?
a) tudo  b) só o `import` (e a senha sai do código)  c) o laço  d) os `except`

<details>
<summary><b>Resposta da 3</b></summary>

**b**. O simulador tem a mesma interface da Netmiko.

</details>

## 🏠 Para casa

- **No encontro seguinte — noite B:** laboratório, sem assunto novo. Os 🎯 que
  ficaram, o chamado resolvido e a [Lista
  14](https://lacouth.github.io/python_telecom-site/listas/lista14/) começada em sala.
- Releia no [capítulo 14 do
  site](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/) as
  seções que ficaram difíceis: o link 📖 de cada bloco leva direto a elas.
- Guarde a pergunta 1 do 🚪: ela abre a noite B.